# Recepción del estreno: críticos vs. usuarios (4 al 13 de abril de 2025)

En este notebook queremos ver cómo recibió el público y la crítica a *A Minecraft Movie* durante la primera semana del estreno. Nos interesa saber cuántas reseñas fueron positivas, mixtas o negativas, y si los críticos y los usuarios opinaron parecido.

Usamos las mismas reseñas de Metacritic que el resto del equipo, pero solo las que caen entre el 4 y el 13 de abril de 2025, que es el estreno.

Para clasificar las reseñas usamos los criterios de Metacritic. En el caso de los críticos, usamos la columna `sentiment` que ya tiene el archivo del equipo. En el caso de los usuarios, la clasificación la hacemos nosotros con el puntaje: de 7 a 10 es positiva, de 4 a 6 es mixta y de 0 a 3 es negativa.

Como las escalas son distintas (los usuarios puntúan de 0 a 10 y los críticos de 0 a 100), vamos a comparar proporciones y no promedios.

## Cargamos los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/reseñas_estreno_metacritic.csv')
print(df.shape)
df.head()

Antes de contar algo, miramos si hay valores vacíos o reseñas repetidas.

In [ ]:
print(df.isnull().sum())
print('Duplicados:', df.duplicated(['tipo','autor','fecha','puntaje_original']).sum())

Las fechas ya están en formato `AAAA-MM-DD`. Nos aseguramos de que todas caigan en el estreno y limpiamos los espacios de los nombres.

In [ ]:
print('Primera fecha:', df.fecha.min(), '| Última fecha:', df.fecha.max())
df['autor'] = [a.strip() for a in df.autor]
df['clasificacion'] = [c.strip() for c in df.clasificacion]
df.clasificacion.value_counts()

## ¿Cuántas reseñas fueron positivas, mixtas y negativas?

In [ ]:
conteo = df.groupby(['tipo','clasificacion']).size().unstack(fill_value=0)
conteo['total'] = conteo.sum(axis=1)
conteo

In [ ]:
porcentaje = (conteo[['Positiva','Mixta','Negativa']].div(conteo['total'], axis=0) * 100).round(1)
porcentaje

Hay 189 reseñas de usuarios, pero solo 11 de críticos en esos días. Con tan pocas reseñas de críticos, sus porcentajes son más una referencia que una conclusión firme.

## ¿Cómo fue cambiando día a día?

Miramos las reseñas de usuarios por día para ver si el entusiasmo se mantuvo o si fue cayendo.

In [ ]:
usuarios = df[df.tipo == 'usuario']
por_dia = usuarios.groupby(['fecha','clasificacion']).size().unstack(fill_value=0)
for c in ['Positiva','Mixta','Negativa']:
    if c not in por_dia.columns:
        por_dia[c] = 0
por_dia = por_dia[['Positiva','Mixta','Negativa']]
por_dia['total'] = por_dia.sum(axis=1)
por_dia

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
por_dia[['Positiva','Mixta','Negativa']].plot(kind='bar', stacked=True, ax=ax,
                                             color=['#2E7D32','#FBC02D','#C62828'])
ax.set_title('Reseñas de usuarios por día durante el estreno')
ax.set_xlabel('Fecha')
ax.set_ylabel('Cantidad de reseñas')
ax.legend(title='Clasificación')
plt.tight_layout()
plt.savefig('../images/estreno_resenias_por_dia.png', dpi=300, bbox_inches='tight')
plt.show()

## Críticos vs. usuarios

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
porcentaje.T.plot(kind='bar', ax=ax, color=['#1565C0','#90CAF9'])
ax.set_title('Clasificación de reseñas en el estreno: críticos vs. usuarios')
ax.set_ylabel('% de reseñas')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../images/estreno_criticos_vs_usuarios.png', dpi=300, bbox_inches='tight')
plt.show()

## Conclusiones

- **Los usuarios fueron mayormente positivos.** Un 50,8 % de las reseñas fueron positivas, un 15,3 % mixtas y un 33,9 % negativas.
- **Los críticos fueron menos entusiastas.** Solo un 18,2 % fue positivo, y casi la mitad (45,5 %) fue mixto. Son pocas reseñas, así que hay que tomarlo con cuidado.
- **El entusiasmo se concentró al principio.** Las positivas de usuarios fueron cerca del 53 % los dos primeros días y bajaron a un 33 % entre el 9 y el 12 de abril. Después del 8 casi no hay reseñas por día, así que no conviene sacar conclusiones de esos días.

## Limitaciones

- Quien escribe una reseña suele tener una opinión fuerte, así que no representa a todo el público.